In [25]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import time 
import numpy as np
import pandas as pd

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

In [2]:
col = ["duration","protocol_type","service","flag","src-bytes","dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins","logged_in","num-compromised","root-shell","su-attempted","num_root","num_file_creation","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_error_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","class_attack","class_score"]
url = "KDDTest+1.csv"
data = pd.read_csv(url,names=col)
data.shape

(22544, 43)

In [3]:
# remove attribute 'difficulty_level'
data.drop(['class_score'],axis=1,inplace=True)
data.shape

(22544, 42)

In [4]:
# changing attack labels to their respective attack class
# total 38 different sub-attacks 
def change_label(df):
  df.class_attack.replace(['apache2','back','land','neptune','mailbomb','pod','processtable','smurf','teardrop','udpstorm','worm'],'Dos',inplace=True)
  df.class_attack.replace(['ftp_write','guess_passwd','httptunnel','imap','multihop','named','phf','sendmail',
       'snmpgetattack','snmpguess','spy','warezclient','warezmaster','xlock','xsnoop'],'R2L',inplace=True)
  df.class_attack.replace(['ipsweep','mscan','nmap','portsweep','saint','satan'],'Probe',inplace=True)
  df.class_attack.replace(['buffer_overflow','loadmodule','perl','ps','rootkit','sqlattack','xterm'],'U2R',inplace=True)


In [5]:
# calling change_label() function
change_label(data)

In [6]:
# distribution of attack classes
data.class_attack.value_counts()

normal    9711
Dos       7460
R2L       2885
Probe     2421
U2R         67
Name: class_attack, dtype: int64

In [7]:
# changing attack labels into two categories 'normal' and 'abnormal'
bin_label = pd.DataFrame(data.class_attack.map(lambda x:'normal' if x=='normal' else 'abnormal'))

In [8]:
# creating a dataframe with binary labels (normal,abnormal)
bin_data_test = data.copy()
bin_data_test['class_attack'] = bin_label

In [9]:
# label encoding (0,1) binary labels (abnormal,normal)
le1_test = preprocessing.LabelEncoder()
enc_label = bin_label.apply(le1_test.fit_transform)
bin_data_test['intrusion'] = enc_label

In [10]:
# one-hot-encoding attack label
bin_data_test = pd.get_dummies(bin_data_test,columns=['protocol_type','service','flag','class_attack'],prefix="",prefix_sep="") 
bin_data_test['class_attack'] = bin_label

In [11]:
bin_data_test

,duration,src-bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num-compromised,...,RSTR,S0,S1,S2,S3,SF,SH,abnormal,normal,class_attack
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,abnormal
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,abnormal
2,2,12983,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,normal
3,0,20,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,abnormal
4,1,0,15,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,0,794,333,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,1,normal
22540,0,317,938,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,1,normal
22541,0,54540,8314,0,0,0,2,0,1,1,...,0,0,0,0,0,1,0,1,0,abnormal
22542,0,42,42,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,normal


In [12]:
import warnings

warnings.filterwarnings("ignore")

X = bin_data_test.iloc[:,0:116] # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = bin_data_test['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)

In [13]:
X_train = preprocessing.scale(X_train)
X_train = preprocessing.normalize(X_train)

In [14]:
X_validation = preprocessing.scale(X_validation)
X_validation = preprocessing.normalize(X_validation)

In [15]:
print(len(X_train), "Training sequences",X_train.shape)
print(len(X_validation), "Validation sequences",X_validation.shape)


18035 Training sequences (18035, 116)
4509 Validation sequences (4509, 116)


In [16]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))
X_validation = np.reshape(X_validation,(X_validation.shape[0],X_validation.shape[1],1))

In [28]:
# Using tanh and sigmoid as activation functions
import time

Model = keras.Sequential([

        keras.layers.Conv2D(96,(4,4),input_shape=(X_train.shape[1],X_train.shape[2],1),activation='relu',padding='same'),
        keras.layers.Conv2D(64,(3,3),activation="relu",padding='same'),
        keras.layers.Conv2D(32,(2,2),activation="relu",padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512,activation="relu"),
        keras.layers.Dense(128,activation="relu"),
        keras.layers.Dense(32,activation="relu"),
        keras.layers.Dense(2,activation="softmax"),
    
    
    ])

Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
start_time = time.time()
#Training the model
Model.fit(X_train, Y_train, epochs=5, batch_size=64) 
Model.summary()

# Final evaluation of the model
scores = Model.evaluate(X_validation, Y_validation, verbose=0)
delta = time.time()- start_time
print("Accuracy: %.2f%%" % (scores[1]*100))
print("Training time: %.2f sec" % (delta))

Epoch 1/5
282/282 [==============================] - 4s 8ms/step - loss: 0.0506 - sparse_categorical_accuracy: 0.9788
Epoch 2/5
282/282 [==============================] - 2s 8ms/step - loss: 0.0019 - sparse_categorical_accuracy: 0.9993
Epoch 3/5
282/282 [==============================] - 2s 9ms/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9993
Epoch 4/5
282/282 [==============================] - 2s 9ms/step - loss: 0.0014 - sparse_categorical_accuracy: 0.9996
Epoch 5/5
282/282 [==============================] - 2s 9ms/step - loss: 2.4769e-04 - sparse_categorical_accuracy: 0.9999
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 116, 1, 96)        1632      
                                                                 
 conv2d_4 (Conv2D)           (None, 116, 1, 64)        55360     
                                                      

In [36]:
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error
                             ,mean_absolute_error,roc_auc_score,confusion_matrix)
from sklearn import metrics

In [41]:
y_pred = Model.predict(X_validation, verbose = 0)
# predict crisp classes for test set
yhat_classes = np.argmax(y_pred,axis=1)
# reduce to 1d array
yhat_probs = y_pred[:, 0]

 
# accuracy: (tp + tn) / (p + n)
accuracy = accuracy_score(Y_validation, yhat_classes)
print('Accuracy: %f' % accuracy)
# precision tp / (tp + fp)
precision = precision_score(Y_validation, yhat_classes)
print('Precision: %f' % precision)
# recall: tp / (tp + fn)
recall = recall_score(Y_validation, yhat_classes)
print('Recall: %f' % recall)
# f1: 2 tp / (2 tp + fp + fn)
f1 = f1_score(Y_validation, yhat_classes)
print('F1 score: %f' % f1)


# ROC AUC
auc = roc_auc_score(Y_validation, yhat_probs)
print('ROC AUC: %f' % auc)
# confusion matrix
matrix = confusion_matrix(Y_validation, yhat_classes)
print(matrix)
# false alaram rate
false_alaram_rate = matrix[1,0]/(matrix[1,0]+matrix[0,0])
print('false alaram rate: %f' % false_alaram_rate)

Accuracy: 0.999778
Precision: 0.999487
Recall: 1.000000
F1 score: 0.999743
ROC AUC: 0.000000
[[2561    1]
 [   0 1947]]
false alaram rate: 0.000000
